# 01. Single-Chain DEX <-> DEX Statistical Arbitrage

> **Goal:** detect and backtest price-divergence opportunities between multiple AMMs on the same chain.

---

## 1. Configuration

```python
# Cell metadata: {"tags"["parameters","hide_input"]}

from src.data_utils    import load_token_data
from src.preprocessing import bucket_1min
from src.smoothing     import ema_smooth
from src.strategy      import zscore_strategy
from src.backtest      import run_backtest, size_full_capital
from src.metrics       import summarize_performance

# ▶️ Which chain and token are we testing?
CHAIN = "SOL"
TOKEN = "Bonk"

# ▶️ Which on‐chain DEXes do we have?
DEXES = {
    "Raydium":   "DezXAZ8z7PnrnRJjz3wXBoRgixCa6xjnB7YaB1pPB263",
    "Orca":      "7bfje1a..."
    # …fill in the rest from your table…
}

# ▶️ Time window
START = "2024-01-01"
END   = "2024-04-30"
```
<details>
<summary>Click to expand</summary>

```python
# Load raw trades for each DEX, stitch together
raw = { name: load_token_data(CHAIN, addr, START, END)
        for name, addr in DEXES.items() }

# 1-minute VWAP buckets
buckets = { name: bucket_1min(df) for name, df in raw.items() }

# Merge into one DataFrame
import pandas as pd
df = pd.concat([
    bkt[["bucket","vwap"]].rename(columns={"vwap":name})
    for name, bkt in buckets.items()
], axis=1).sort_index()

# Smooth out spikes
for name in DEXES:
    df[name + "_s"] = ema_smooth(df[name], span=5)
```

</details>

In [15]:
import importlib
import utils

importlib.reload(utils)

<module 'utils' from '/Users/jakkie/Dev/dex-statarb-research/src/utils.py'>

In [2]:
import os, sys
proj_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
print(proj_root)
src_dir   = os.path.join(proj_root, "src")
sys.path.insert(0, src_dir)

/Users/jakkie/Dev/dex-statarb-research


In [17]:
utils.load_data('DYDX', 'FARTCOIN')

s3://iamjakkie-public/dydx/candles/FARTCOIN,RAYDIUM,9BB6NFECJBCTNNLFKO2FQVQBQ8HHM13KCYYCDQBGPUMP-USD.parquet
